# Autograd

Autograd is PyTorch's automatic differentiation system. It tracks all operations performed on tensors with requires_grad=True and automatically computes gradients (derivatives) when backward() is called.

In [1]:
import torch
import warnings
warnings.filterwarnings("ignore")

e:\PyTorch Practice\PyTorch\venv\Lib\site-packages\torch\_subclasses\functional_tensor.py:362: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


- **Leaf tensors** are tensors that are created directly by the user and are not the result of another operation. They often serve as the inputs or learnable parameters of a computation and can have requires_grad=True.
- **Root tensors** are the final output tensors (such as a loss value) from which backpropagation is started by calling .backward().

In [2]:
# Leaf tensor (created directly by the user)
a = torch.tensor(2.0, requires_grad=True)

# Intermediate computation
b = a * 3

# Root/output tensor
c = b + 4

# Compute gradients
c.backward()

print(a.grad)  # Output: tensor(3.)

# Explanation:
# c = 3a + 4
# dc/da = 3
# So the gradient stored in a.grad is 3.

tensor(3.)


In [3]:
# Create a scalar tensor with the value 3.0.
# requires_grad=True tells PyTorch to track all operations performed
# on this tensor so that gradients (derivatives) can be computed later.
x = torch.tensor(3.0, requires_grad=True)
print(x)

# Compute y = x^2.
# Since x = 3.0:
# y = 3^2 = 9
#
# PyTorch records this operation in its computation graph.
# Mathematically:
#     y = x²
#
# The derivative of y with respect to x is:
#     dy/dx = 2x
y = x ** 2
print(y)

# Perform backpropagation.
# This computes the derivative of y with respect to x.
#
# Since:
#     y = x²
#     dy/dx = 2x
#
# and x = 3, the derivative becomes:
#     dy/dx = 2 * 3 = 6
#
# The computed gradient is stored in x.grad.
y.backward()

# Display the computed gradient.
# Expected output:
# tensor(6.)
#
# This means:
#     d(x²)/dx evaluated at x = 3 is 6.
print(x.grad)

tensor(3., requires_grad=True)
tensor(9., grad_fn=<PowBackward0>)
tensor(6.)


In [4]:
import torch

# Create a scalar tensor with the value 3.0.
# Setting requires_grad=True tells PyTorch to keep track of all
# operations involving x so that gradients can be computed automatically.
x = torch.tensor(3.0, requires_grad=True)
print(x)

# Compute y = x².
# Since x = 3:
#     y = 3² = 9
#
# PyTorch records this operation in its computation graph because
# x requires gradients.
y = x ** 2
print(y)

# Compute z = sin(y).
# Since y = 9:
#     z = sin(9)
#
# The overall function is now:
#     z = sin(x²)
#
# PyTorch extends the computation graph to include this operation.
z = torch.sin(y)
print(z)

# Compute the gradient of z with respect to x.
#
# Using the chain rule:
#
#     z = sin(y)
#     y = x²
#
# First derivative:
#     dz/dy = cos(y)
#
# Second derivative:
#     dy/dx = 2x
#
# Therefore:
#     dz/dx = dz/dy * dy/dx
#            = cos(y) * 2x
#            = cos(x²) * 2x
#
# At x = 3:
#     dz/dx = cos(9) * 2 * 3
#            = 6 * cos(9)
#
# This value is automatically computed by PyTorch and stored in x.grad.
z.backward()

# Print the computed gradient.
# The value should be approximately:
#     6 * cos(9) ≈ -5.4668
#
# So the output will be close to:
# tensor(-5.4668)
print(x.grad)

tensor(3., requires_grad=True)
tensor(9., grad_fn=<PowBackward0>)
tensor(0.4121, grad_fn=<SinBackward0>)
tensor(-5.4668)


# Visualize the computation graph (nodes + edges)

In [14]:
import torch
from torchviz import make_dot
from datetime import datetime

x = torch.tensor(3.0, requires_grad=True)

y = x ** 2
z = torch.sin(y)

print(z.grad_fn)
print(z.grad_fn.next_functions)

# Create graph visualization
dot = make_dot(z, params={"x": x})

# Create clean timestamp
timestamp = datetime.now().strftime("%Y-%m-%d__%H-%M-%S")

filename = f"graphs/sin_square_graph__{timestamp}"

dot.render(filename, format="png", cleanup=True)

((<PowBackward0 object at 0x000002827EBCBC70>, 0),)


'graphs\\sin_square_graph__2026-06-11__07-35-57.png'

In [15]:
import torch

# -----------------------------
# VECTOR TENSOR EXAMPLE
# -----------------------------

# Create a vector (1D tensor) with requires_grad=True
# This means PyTorch will track operations on each element
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
print("x:", x)

# Define a simple function:
# y = x^2 (element-wise square)
y = x ** 2
print("y:", y)

# Reduce output to a scalar (important for backward())
# We sum all elements so we can call backward()
z = y.sum()
print("z (sum of y):", z)

# Backpropagation
# dz/dx = 2x (element-wise)
z.backward()

# Gradient of each element in x
print("x.grad:", x.grad)

# Expected:
# tensor([2.0, 4.0, 6.0])

x: tensor([1., 2., 3.], requires_grad=True)
y: tensor([1., 4., 9.], grad_fn=<PowBackward0>)
z (sum of y): tensor(14., grad_fn=<SumBackward0>)
x.grad: tensor([2., 4., 6.])


In [16]:
import torch

# -----------------------------
# SCALAR + VECTOR + GRAD RESET
# -----------------------------

# Create a vector tensor
x = torch.tensor([2.0, 4.0], requires_grad=True)
print("Initial x:", x)

# ---- FIRST BACKWARD PASS ----

y = x * 3
z = y.sum()  # scalar output required for backward()

z.backward()

# Gradient after first backward
print("x.grad after 1st backward:", x.grad)

# Expected:
# dz/dx = 3 → [3, 3]

# ---- IMPORTANT: GRADIENT RESET ----
# If we do NOT reset, gradients will accumulate
x.grad.zero_()

print("x.grad after zero_():", x.grad)

# ---- SECOND BACKWARD PASS ----

# New computation
y2 = x ** 2
z2 = y2.sum()

z2.backward()

# New gradients
print("x.grad after 2nd backward:", x.grad)

# Expected:
# dz/dx = 2x → [4, 8]

Initial x: tensor([2., 4.], requires_grad=True)
x.grad after 1st backward: tensor([3., 3.])
x.grad after zero_(): tensor([0., 0.])
x.grad after 2nd backward: tensor([4., 8.])


1. requires_grad_(False) (turn OFF gradient tracking)

In [17]:
import torch

# -----------------------------
# requires_grad_(False)
# -----------------------------

# Create a tensor that tracks gradients
x = torch.tensor([2.0, 3.0], requires_grad=True)
print("Original x:", x)

# Turn OFF gradient tracking in-place
# This modifies the same tensor
x.requires_grad_(False)

print("After requires_grad_(False):", x)

# Now PyTorch will NOT track operations on x
y = x * 2

# This will NOT compute gradients
# because x is no longer part of computation graph
print("y:", y)

# If we try:
# y.backward()  ❌ ERROR (no grad tracking)

Original x: tensor([2., 3.], requires_grad=True)
After requires_grad_(False): tensor([2., 3.])
y: tensor([4., 6.])


2. detach() (creates a new tensor without graph)

In [18]:
import torch

# -----------------------------
# detach()
# -----------------------------

x = torch.tensor([2.0, 3.0], requires_grad=True)
print("x:", x)

# Create computation
y = x * 2

# detach() creates a NEW tensor
# It shares same memory BUT is removed from computation graph
y_detached = y.detach()

print("y:", y)
print("y_detached:", y_detached)

# Modify detached tensor
z = y_detached * 3

# z is NOT connected to x
# so gradients will NOT flow back to x
print("z:", z)

# If we try backward through z:
# z.backward() ❌ ERROR unless scalar reduction is applied,
# but even then x.grad will remain None

x: tensor([2., 3.], requires_grad=True)
y: tensor([4., 6.], grad_fn=<MulBackward0>)
y_detached: tensor([4., 6.])
z: tensor([12., 18.])


3. torch.no_grad() (temporary context disabling gradients)

In [19]:
import torch

# -----------------------------
# torch.no_grad()
# -----------------------------

x = torch.tensor([2.0, 3.0], requires_grad=True)
print("x:", x)

# Everything inside this block does NOT track gradients
with torch.no_grad():

    # Even though x normally requires gradients,
    # operations here are NOT added to computation graph
    y = x * 2
    z = y ** 2

    print("y (no_grad):", y)
    print("z (no_grad):", z)

# Outside the block, gradient tracking is restored
w = x * 3

print("w (tracked again):", w)

# Backprop still works for tracked part
loss = w.sum()
loss.backward()

print("x.grad:", x.grad)

x: tensor([2., 3.], requires_grad=True)
y (no_grad): tensor([4., 6.])
z (no_grad): tensor([16., 36.])
w (tracked again): tensor([6., 9.], grad_fn=<MulBackward0>)
x.grad: tensor([3., 3.])
